# Module 12 Guided Lab: Introduction to Predictive Modeling with Linear Regression

**BAN 6003: Data Management and Analytics Integration**

In the previous module, we checked and documented an ABT. Now we begin applying analytics to an ABT.

This module introduces supervised predictive modeling through a simple linear regression workflow.

The main idea is:

> Use known variables, called features, to predict a continuous outcome, called the target.

For this lab, we will use the `nycflights13` data to predict arrival delay in minutes.

## Lab Learning Goals

By the end of this lab, you should be able to:

1. Explain the purpose of supervised predictive modeling.
2. Distinguish features and target variables.
3. Split data into training and testing sets.
4. Fit a basic linear regression model using Scikit-learn.
5. Generate predictions.
6. Calculate simple model metrics such as MAE and RMSE.
7. Interpret model outputs cautiously.

This is an introductory modeling lab. The goal is not to build the perfect model. The goal is to understand the basic workflow.

## Business Scenario

Imagine an airline operations team wants to estimate arrival delay.

A simple modeling question could be:

> Given information about a flight, can we estimate its arrival delay?

For this demo, we will use a flight-level ABT and predict:

- **Target variable:** `arr_delay`
- **Features:** `dep_delay`, `distance`, `air_time`, and `month`

Important note: This is a teaching example. In a real project, we must carefully ask when each feature becomes available. For example, `dep_delay` is only known after departure, and `air_time` may not be known until after the flight. That means the business timing of the prediction matters.

## 0. Setup

We will use:

- `pandas` for data handling
- `matplotlib` for basic visualization
- `scikit-learn` for modeling
- `nycflights13` for data

### Package setup note

If you are using **GitHub Codespaces**, the required packages should already be installed from `requirements.txt` when the Codespace is created. You usually do not need to run any `%pip install` command.

If you are running the repository on your **local computer** and an import fails, uncomment and run the `%pip install` line(s) in the setup cell below, then rerun the imports.



In [ ]:
# Codespaces: nycflights13 and scikit-learn are installed from requirements.txt.
# Local only: if the imports below fail, uncomment and run the needed line(s):
# %pip install nycflights13
# %pip install -q scikit-learn

import nycflights13
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 1. Load the Flight ABT

For this introductory lab, we will create a small flight-level ABT from `nycflights13.flights`.

One row represents one flight.

In [ ]:
flights = nycflights13.flights.copy()

flight_abt = flights[[
    "month",
    "dep_delay",
    "arr_delay",
    "distance",
    "air_time"
]].copy()

flight_abt.head()

In [ ]:
flight_abt.shape

## 2. Check Missing Values

Scikit-learn's basic `LinearRegression` model cannot use rows with missing values.

For this first modeling lab, we will use a simple approach: drop rows with missing values in the selected columns.

Later, more advanced workflows may use imputation or more careful missing-value strategies.

In [ ]:
flight_abt.isna().sum()

In [ ]:
model_data = flight_abt.dropna().copy()

flight_abt.shape, model_data.shape

### Quick interpretation

Dropping rows is simple, but it is a modeling decision. In a real project, we should document how many rows were removed and whether that could bias the analysis.

## 3. Define Target and Features

In supervised learning, we usually separate:

- `y`: the target variable we want to predict
- `X`: the feature variables used to make the prediction

For this lab:

- `y = arr_delay`
- `X = dep_delay, distance, air_time, month`

In [ ]:
target = "arr_delay"
features = ["dep_delay", "distance", "air_time", "month"]

X = model_data[features]
y = model_data[target]

X.head()

In [ ]:
y.head()

### Your Turn 1

Write one sentence explaining the difference between the target variable and the feature variables in this lab.

**Your answer:**  
Type your answer here.

## 4. Visualize a Simple Relationship

Before modeling, it is helpful to look at one relationship.

Here, we plot `dep_delay` against `arr_delay`.

We will use a sample of the data so the plot is easier to read.

In [ ]:
plot_sample = model_data.sample(n=2000, random_state=42)

plt.figure(figsize=(8, 5))
plt.scatter(plot_sample["dep_delay"], plot_sample["arr_delay"], alpha=0.4)
plt.xlabel("Departure delay (minutes)")
plt.ylabel("Arrival delay (minutes)")
plt.title("Departure Delay vs Arrival Delay")
plt.grid(True, alpha=0.25)
plt.show()

The relationship is not perfect, but we should see a positive pattern: flights that leave later often arrive later.

Linear regression tries to summarize this kind of relationship with a straight-line pattern.

## 5. Train/Test Split

We split data into:

- **Training data:** used to fit the model
- **Testing data:** used to evaluate the model on data it did not train on

This helps us avoid judging the model only on the data it already saw.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

Important arguments:

- `test_size=0.25`: 25% of the data is used for testing.
- `random_state=42`: makes the split reproducible.

### Your Turn 2

Change the `test_size` in the cell below to 0.30 and run the split. Compare the new training and testing sizes.

Then change it back to 0.25 for the rest of the lab.

In [ ]:
# Your Turn 2
# Try a 30% test split here.

X_train_demo, X_test_demo, y_train_demo, y_test_demo = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

X_train_demo.shape, X_test_demo.shape

## 6. Fit a Linear Regression Model

Now we create a `LinearRegression` object and fit it using the training data.

In [ ]:
model = LinearRegression()

model.fit(X_train, y_train)

print("Model fitted.")

The model has learned coefficients. Each coefficient is connected to one feature.

For a basic interpretation, a positive coefficient means the predicted arrival delay increases as that feature increases, holding the other features constant.

Be cautious: this is a prediction model, not automatically a causal explanation.

In [ ]:
coefficients = pd.DataFrame({
    "feature": features,
    "coefficient": model.coef_
})

coefficients

In [ ]:
model.intercept_

### Your Turn 3

Which feature has the largest positive coefficient? Write one cautious interpretation.

Avoid causal language such as "causes" unless the research design supports it.

**Your answer:**  
Type your answer here.

## 7. Generate Predictions

Now we use the fitted model to predict arrival delay for the test data.

In [ ]:
y_pred = model.predict(X_test)

predictions = pd.DataFrame({
    "actual_arr_delay": y_test,
    "predicted_arr_delay": y_pred
})

predictions.head()

## 8. Evaluate the Model

We will use three simple metrics:

- **MAE:** mean absolute error. Average size of prediction error in minutes.
- **RMSE:** root mean squared error. Similar to MAE, but penalizes large errors more.
- **R-squared:** proportion of variation explained by the model, roughly speaking.

For business communication, MAE is often intuitive because it is in the original unit: minutes.

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

metrics = pd.DataFrame({
    "metric": ["MAE", "RMSE", "R-squared"],
    "value": [mae, rmse, r2]
})

metrics

### Your Turn 4

Write a 2–3 sentence interpretation of the model performance.

Use plain language. For example, explain what MAE means in minutes.

**Your answer:**  
Type your answer here.

## 9. Plot Actual vs Predicted Values

A model can have metrics, but we should also look at the predictions visually.

If predictions were perfect, points would fall exactly on the diagonal line.

In [ ]:
plot_results = predictions.sample(n=2000, random_state=42)

plt.figure(figsize=(8, 5))
plt.scatter(plot_results["actual_arr_delay"], plot_results["predicted_arr_delay"], alpha=0.4)
plt.xlabel("Actual arrival delay (minutes)")
plt.ylabel("Predicted arrival delay (minutes)")
plt.title("Actual vs Predicted Arrival Delay")
plt.grid(True, alpha=0.25)

# Add a reference diagonal line
min_val = min(plot_results["actual_arr_delay"].min(), plot_results["predicted_arr_delay"].min())
max_val = max(plot_results["actual_arr_delay"].max(), plot_results["predicted_arr_delay"].max())
plt.plot([min_val, max_val], [min_val, max_val], linewidth=2)

plt.show()

## 10. Prediction vs Explanation

Linear regression can be used for prediction and sometimes for explanation, but these are not the same.

For this lab:

- We are mainly learning the prediction workflow.
- Coefficients are useful, but we should interpret them cautiously.
- We should not claim causality from this simple model.
- We should consider timing and data leakage.

A good business interpretation is clear about what the model can and cannot support.

## 11. Final Practice: Fit Your Own Simple Regression

Create a simpler model using only one feature:

- Feature: `dep_delay`
- Target: `arr_delay`

Then fit the model, generate predictions, and calculate MAE and RMSE.

In [ ]:
# Final Practice Step 1
# Create X_simple and y_simple.

In [ ]:
# Final Practice Step 2
# Split into training and testing data.

In [ ]:
# Final Practice Step 3
# Fit LinearRegression.

In [ ]:
# Final Practice Step 4
# Generate predictions and calculate MAE and RMSE.

### Final Reflection

Write 5–7 sentences answering:

1. What was the target variable?
2. What features did the main model use?
3. What does the train/test split help us do?
4. Which metric was easiest to interpret?
5. What is one limitation of this simple model?
6. How might you apply this workflow to your project ABT?

**Your reflection:**  
Type your answer here.

## 12. Save Your Work

Before submitting:

1. Save this notebook.
2. Make sure all cells run.
3. Complete all Your Turn sections.
4. Complete the final practice and reflection.
5. Commit and push your work.

Suggested commands:

```bash
git add .
git commit -m "Complete Module 12 linear regression lab"
git push
```